In [1]:
!pip install -q transformers accelerate huggingface_hub gradio soundfile librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.1 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which

In [2]:
!pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 112.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 30.0 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


### Task 1 — Explore Hugging Face

In [3]:
from huggingface_hub import login,whoami
import getpass

In [4]:
token = getpass.getpass("Enter HF Token: ")
login(token)

Enter HF Token:  ········


In [5]:
print(whoami())

{'type': 'user', 'id': '6a2e763d34eb24c077189020', 'name': 'AsafKhan', 'fullname': 'Asaf Khan', 'email': 'p233048@pwr.nu.edu.pk', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1782864000, 'isPro': False, 'avatarUrl': '/avatars/a9d4e7b1771b147016ec5f318cf6774f.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'begin', 'role': 'read', 'createdAt': '2026-06-14T09:40:59.466Z'}}}


In [6]:
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

model_id = "microsoft/VibeVoice-ASR-HF"

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    device_map="auto"
)

processor_config.json:   0%|          | 0.00/537 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

[transformers] `VibeVoiceAsrProcessor` defines `feature_extractor_class = 'VibeVoiceAcousticTokenizerFeatureExtractor'`, which is deprecated. Register the correct mapping in `AutoFeatureExtractor` instead.


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/714 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/901 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

In [7]:
audio_path = "/kaggle/input/datasets/kngzmn/recording/Recording.m4a"

In [8]:
inputs = processor.apply_transcription_request(
    audio=audio_path
).to(model.device, model.dtype)

output_ids = model.generate(**inputs)

generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]

text = processor.decode(generated_ids)

print(text)

[transformers] Both `max_new_tokens` (=32768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


['<|im_start|>assistant\n[{"Start":0,"End":14.0,"Speaker":0,"Content":"Hello, how are you? This is just a demonstration of how the AI model is going to work. It is probably working, I think, fine, because I can see how they work."},{"Start":14.0,"End":21.74,"Speaker":0,"Content":"How the words coming out of my mouth are translated perfectly and written. So, that\'s okay."}]<|im_end|>\n<|endoftext|>']


### Task 2 — Work With a Real Datase

In [ ]:
# PART A — Provided Dataset: Pakistan Air Quality
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# We generate a realistic Pakistan AQI dataset
# In Week 2 you will load real datasets directly from Kaggle API
np.random.seed(42)
n = 500

cities = np.random.choice(['Lahore', 'Karachi', 'Peshawar', 'Islamabad', 'Quetta'], n)
months = np.random.choice(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                           'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], n)
pm25 = np.where(cities == 'Lahore',
    np.random.normal(180, 40, n),
    np.where(cities == 'Karachi',
        np.random.normal(150, 35, n),
        np.where(cities == 'Peshawar',
            np.random.normal(160, 38, n),
            np.random.normal(90, 25, n))))

pm25 = np.clip(pm25, 10, 400)
temperature = np.random.normal(28, 10, n)
humidity = np.random.normal(55, 20, n)
wind_speed = np.random.normal(12, 5, n)

# AQI Category based on PM2.5
def categorize_aqi(pm):
    if pm < 50: return 'Good'
    elif pm < 100: return 'Moderate'
    elif pm < 150: return 'Unhealthy for Sensitive Groups'
    else: return 'Unhealthy'

aqi_category = [categorize_aqi(p) for p in pm25]
is_unhealthy = (pm25 >= 150).astype(int)

df = pd.DataFrame({
    'City': cities,
    'Month': months,
    'PM2.5': pm25.round(1),
    'Temperature': temperature.round(1),
    'Humidity': humidity.round(1),
    'Wind Speed': wind_speed.round(1),
    'AQI Category': aqi_category,
    'Unhealthy': is_unhealthy
})

print('Pakistan Air Quality Dataset')
print('=' * 50)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nFirst 5 rows:')
print(df.head())
print(f'\nBasic Statistics:')
print(df.describe().round(2))

In [ ]:
# Visualize the data
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# PM2.5 by city
city_avg = df.groupby('City')['PM2.5'].mean().sort_values(ascending=False)
axes[0].bar(city_avg.index, city_avg.values,
            color=['#e74c3c', '#e67e22', '#f39c12', '#27ae60', '#2980b9'])
axes[0].axhline(y=150, color='red', linestyle='--', label='Unhealthy threshold')
axes[0].set_title('Average PM2.5 by City', fontweight='bold')
axes[0].set_ylabel('PM2.5 (ug/m3)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=15)

# AQI distribution
aqi_counts = df['AQI Category'].value_counts()
colors = ['#27ae60', '#f39c12', '#e67e22', '#e74c3c']
axes[1].pie(aqi_counts.values, labels=aqi_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('AQI Category Distribution', fontweight='bold')

# PM2.5 vs Wind Speed
scatter = axes[2].scatter(df['Wind Speed'], df['PM2.5'],
                          c=df['Unhealthy'], cmap='RdYlGn_r',
                          alpha=0.5, s=20)
axes[2].set_title('Wind Speed vs PM2.5', fontweight='bold')
axes[2].set_xlabel('Wind Speed (km/h)')
axes[2].set_ylabel('PM2.5 (ug/m3)')
axes[2].axhline(y=150, color='red', linestyle='--', alpha=0.5)
plt.colorbar(scatter, ax=axes[2], label='Unhealthy (1=Yes)')

plt.suptitle('Pakistan Air Quality Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Lahore and Peshawar consistently have the worst air quality.')
print('Higher wind speed generally correlates with lower PM2.5 — wind disperses pollutants.')
print('This is a real public health problem. AI can help predict bad air days in advance.')

In [ ]:
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Separate encoder for each categorical column
city_encoder = LabelEncoder()
city_encoder.fit(df['City'])

month_encoder = LabelEncoder()
month_encoder.fit(df['Month'])

df_model = df.copy()
df_model['City_encoded'] = city_encoder.transform(df_model['City'])
df_model['Month_encoded'] = month_encoder.transform(df_model['Month'])

features = ['City_encoded', 'Month_encoded', 'Temperature', 'Humidity', 'Wind Speed']
X = df_model[features]
y = df_model['Unhealthy']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)

print('Decision Tree — Predicting Unhealthy Air Quality Days')
print('=' * 55)
print(f'Training Accuracy: {accuracy_score(y_train, dt.predict(X_train))*100:.1f}%')
print(f'Test Accuracy:     {accuracy_score(y_test, dt.predict(X_test))*100:.1f}%')
print()
print(classification_report(y_test, dt.predict(X_test),
      target_names=['Healthy', 'Unhealthy']))

plt.figure(figsize=(20, 7))
plot_tree(dt,
          feature_names=features,
          class_names=['Healthy', 'Unhealthy'],
          filled=True, rounded=True, fontsize=9)
plt.title('Decision Tree — Pakistan Air Quality Prediction', fontsize=13)
plt.tight_layout()
plt.show()

# Real prediction
test_day = pd.DataFrame({
    'City_encoded': [city_encoder.transform(['Lahore'])[0]],
    'Month_encoded': [month_encoder.transform(['Jan'])[0]],
    'Temperature': [35],
    'Humidity': [70],
    'Wind Speed': [5]
})
pred = dt.predict(test_day)[0]
prob = dt.predict_proba(test_day)[0]
print(f'Lahore — Hot day, high humidity, low wind:')
print(f'Prediction: {"Unhealthy" if pred == 1 else "Healthy"}')
print(f'Confidence: {max(prob)*100:.1f}%')
print()
print('This is the kind of model that could power a real air quality warning system.')
print('Same technique. Real data. Real impact.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/kngzmn/cardataset/CarPrice_Assignment.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df = df.drop(['car_ID', 'CarName'], axis=1)

In [ ]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['price'], kde=True)
plt.title("Distribution of Car Prices")
plt.show()

In [ ]:
X = df.drop('price', axis=1)
y = df['price']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("R2 Score:", r2)

**Dataset you chose:** The dataset I choosed is about cars and all it's related data

**What problem it could help solve:** This model could be very help for a client who needs a car with specific features and the model would predicit it price helping the client in the budget prespective

**What model you trained and what it predicted:** I trained a linear regression model and it predicts the price of a car

**One interesting insight:** This could be very helpful for those buying a car and wants to know the correct price instead of relying on other people choice

### Task 3 — Break Something

In [15]:
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

sentiment = pipeline('sentiment-analysis')

experiments = [
    "Wow amazing weather 🌞🔥 it’s sooo cool yaar, bilkul thand lag rahi hai 😂 bohat perfect heatwave ha",
    "The weather is beautifully dangerous, pleasantly freezing, and wonderfully disastrous for outdoor survival",
    "Yeah I absolutly love the way you dress 😂"
]

for text in experiments:
    result = sentiment(text)[0]
    print(f'{result["label"]} ({result["score"]*100:.1f}%) — {text}')

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

POSITIVE (99.9%) — Wow amazing weather 🌞🔥 it’s sooo cool yaar, bilkul thand lag rahi hai 😂 bohat perfect heatwave ha
POSITIVE (95.2%) — The weather is beautifully dangerous, pleasantly freezing, and wonderfully disastrous for outdoor survival
POSITIVE (100.0%) — Yeah I absolutly love the way you dress 😂


---
 
### Experiment 1
> *"Wow amazing weather 🥵🔥 it's sooo cool yaar, bilkul thand lag rahi hai 😂 bohat perfect heatwave ha"*
 
**Result: POSITIVE (99.9%) — ❌ WRONG**
 
- The sentence is **sarcastic** — saying "amazing" and "cool" while describing a heatwave
- The model picks up on surface-level positive words (`amazing`, `cool`, `perfect`) and ignores the contradictory context
- **Code-mixed language** (Urdu/Hindi: *thand lag rahi hai*, *bohat*) is completely ignored — the model was trained on English only, so it misses the sarcastic Urdu punchline
- Emojis (🥵🔥😂) signal irony but the model has no reliable emoji-to-sentiment mapping
---
 
### Experiment 2
> *"The weather is beautifully dangerous, pleasantly freezing, and wonderfully disastrous for outdoor survival"*
 
**Result: POSITIVE (95.2%) — ❌ WRONG**
 
- A classic case of **oxymoronic sarcasm** — pairing positive adverbs (`beautifully`, `pleasantly`, `wonderfully`) with negative nouns (`dangerous`, `freezing`, `disastrous`)
- DistilBERT's sentiment head latches onto the **adverbs** more than the **nouns they modify**
- This requires understanding **semantic contradiction**, which a fine-tuned SST-2 model isn't trained for
---
 
### Experiment 3
> *"Yeah I absolutly love the way you dress 🙄"*
 
**Result: POSITIVE (100.0%) — ❌ WRONG**
 
- Pure **sarcasm with a dismissive tone** — "love" is used sarcastically
- The model sees `love` and immediately goes POSITIVE with full confidence
- The 🙄 emoji is a strong sarcasm signal that the model completely ignores
- No sarcasm-labeled data exists in SST-2, so the model has never learned this pattern
---
 
### 🔍 Root Cause (All Three)
 
> `distilbert-base-uncased-finetuned-sst-2` was trained on **movie reviews** (SST-2 dataset) — straightforward positive/negative opinions.
> It has never seen **sarcasm**, **code-mixed language**, or **oxymoronic constructions**,
> so it always falls back to **keyword-level sentiment**.